# Task 3: Database Engineering — PostgreSQL
## Storing Processed Reviews in a Relational Schema

**Objective:** Design and populate a PostgreSQL database (`bank_reviews`) that persistently stores
the scraped and analysed review data, following production-grade relational design principles.

### Schema Design

```
banks (bank_id PK, bank_name, app_name)
  └── reviews (review_id PK, bank_id FK, review_text, rating, review_date,
               sentiment_label, sentiment_score, identified_theme, source)
```

The schema is **third-normal form (3NF)**: bank metadata lives in one table; all reviews reference
it by `bank_id` to avoid repeating bank names across 1,800+ rows.

In [ ]:
import psycopg2
import pandas as pd
import os
from psycopg2.extras import execute_values

os.chdir('/home/code0053/fintech-review-analytics')

DB_CONFIG = {
    "host":     "localhost",
    "database": "bank_reviews",
    "user":     "bankuser",
    "password": "bankpass123",
    "port":     5432,
}

try:
    conn = psycopg2.connect(**DB_CONFIG)
    print("Connected to PostgreSQL successfully!")
    print(f"Database : {conn.get_dsn_parameters()['dbname']}")
    print(f"Host     : {conn.get_dsn_parameters()['host']}")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")
    print()
    print("To create the database and user, run:")
    print("  sudo -u postgres psql -c \"CREATE DATABASE bank_reviews;\"")
    print("  sudo -u postgres psql -c \"CREATE USER bankuser WITH PASSWORD 'bankpass123';\"")
    print("  sudo -u postgres psql -c \"GRANT ALL PRIVILEGES ON DATABASE bank_reviews TO bankuser;\"")

## 1. Create Tables

The SQL below creates:
- **`banks`** — master table for the three Ethiopian banks
- **`reviews`** — fact table for all scraped and analysed reviews

`ON DELETE CASCADE` on the foreign key ensures review rows are removed if a bank row is deleted,
maintaining referential integrity.

In [ ]:
CREATE_BANKS_SQL = """
CREATE TABLE IF NOT EXISTS banks (
    bank_id   SERIAL PRIMARY KEY,
    bank_name VARCHAR(100) NOT NULL UNIQUE,
    app_name  VARCHAR(200)
);
"""

CREATE_REVIEWS_SQL = """
CREATE TABLE IF NOT EXISTS reviews (
    review_id        SERIAL PRIMARY KEY,
    bank_id          INTEGER NOT NULL REFERENCES banks(bank_id) ON DELETE CASCADE,
    review_text      TEXT,
    rating           SMALLINT CHECK (rating BETWEEN 1 AND 5),
    review_date      DATE,
    sentiment_label  VARCHAR(20),
    sentiment_score  NUMERIC(6, 4),
    identified_theme VARCHAR(100),
    source           VARCHAR(50) DEFAULT 'Google Play'
);
"""

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    cur.execute(CREATE_BANKS_SQL)
    cur.execute(CREATE_REVIEWS_SQL)
    conn.commit()
    cur.close()
    conn.close()
    print("Tables created (or already exist).")
except Exception as e:
    print(f"Error: {e}")

## 2. Insert Bank Metadata

In [ ]:
BANK_METADATA = [
    ("Commercial Bank of Ethiopia", "CBE Mobile Banking"),
    ("Bank of Abyssinia",           "BOA Mobile Banking"),
    ("Dashen Bank",                 "Dashen Super App"),
]

INSERT_BANK_SQL = """
INSERT INTO banks (bank_name, app_name)
VALUES (%s, %s)
ON CONFLICT (bank_name) DO NOTHING;
"""

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    for bank_name, app_name in BANK_METADATA:
        cur.execute(INSERT_BANK_SQL, (bank_name, app_name))
    conn.commit()
    cur.execute("SELECT * FROM banks;")
    rows = cur.fetchall()
    print("Banks table:")
    for row in rows:
        print(f"  {row}")
    cur.close()
    conn.close()
except Exception as e:
    print(f"Error: {e}")

## 3. Insert Review Data

In [ ]:
df = pd.read_csv('data/raw/bank_reviews_sentiment.csv')
df['review_date'] = pd.to_datetime(df['date'], errors='coerce').dt.date

print(f"Loaded {len(df)} reviews for insertion.")
print(df[['review', 'rating', 'date', 'bank', 'sentiment_label', 'identified_theme']].head(3).to_string())

In [ ]:
INSERT_REVIEW_SQL = """
INSERT INTO reviews
    (bank_id, review_text, rating, review_date,
     sentiment_label, sentiment_score, identified_theme, source)
VALUES %s
ON CONFLICT DO NOTHING;
"""

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()

    # Fetch bank_id lookup
    cur.execute("SELECT bank_name, bank_id FROM banks;")
    bank_id_map = {row[0]: row[1] for row in cur.fetchall()}

    records = []
    for _, row in df.iterrows():
        bank_id = bank_id_map.get(row['bank'])
        if bank_id is None:
            continue
        records.append((
            bank_id,
            str(row['review'])[:2000] if pd.notna(row['review']) else None,
            int(row['rating']),
            row['review_date'],
            row.get('sentiment_label'),
            float(row.get('sentiment_score', 0)),
            row.get('identified_theme'),
            'Google Play',
        ))

    execute_values(cur, INSERT_REVIEW_SQL, records)
    conn.commit()
    print(f"Inserted {len(records)} reviews into the database.")
    cur.close()
    conn.close()
except Exception as e:
    print(f"Error during insert: {e}")

## 4. Verification Queries

Confirm data integrity after insertion.

In [ ]:
VERIFICATION_QUERIES = {
    "Count reviews per bank": """
        SELECT b.bank_name, COUNT(r.review_id) AS review_count
        FROM banks b
        LEFT JOIN reviews r ON b.bank_id = r.bank_id
        GROUP BY b.bank_name
        ORDER BY review_count DESC;
    """,
    "Average rating per bank": """
        SELECT b.bank_name, ROUND(AVG(r.rating), 2) AS avg_rating
        FROM banks b
        JOIN reviews r ON b.bank_id = r.bank_id
        GROUP BY b.bank_name
        ORDER BY avg_rating DESC;
    """,
    "Null check on key columns": """
        SELECT
            COUNT(*) FILTER (WHERE review_text IS NULL) AS null_text,
            COUNT(*) FILTER (WHERE rating IS NULL)      AS null_rating,
            COUNT(*) FILTER (WHERE review_date IS NULL) AS null_date,
            COUNT(*) FILTER (WHERE sentiment_label IS NULL) AS null_sentiment
        FROM reviews;
    """,
    "Sentiment distribution": """
        SELECT b.bank_name, r.sentiment_label, COUNT(*) AS n
        FROM reviews r
        JOIN banks b ON r.bank_id = b.bank_id
        GROUP BY b.bank_name, r.sentiment_label
        ORDER BY b.bank_name, n DESC;
    """,
}

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    for title, sql in VERIFICATION_QUERIES.items():
        print(f"\n=== {title} ===")
        cur.execute(sql)
        cols = [desc[0] for desc in cur.description]
        rows = cur.fetchall()
        result = pd.DataFrame(rows, columns=cols)
        print(result.to_string(index=False))
    cur.close()
    conn.close()
except Exception as e:
    print(f"Query error: {e}")

## 5. Schema Export

Run the block below to save the schema as a SQL file for version control.

In [ ]:
schema_sql = '''-- Schema: bank_reviews database
-- Generated for fintech-review-analytics Task 3

CREATE TABLE IF NOT EXISTS banks (
    bank_id   SERIAL PRIMARY KEY,
    bank_name VARCHAR(100) NOT NULL UNIQUE,
    app_name  VARCHAR(200)
);

CREATE TABLE IF NOT EXISTS reviews (
    review_id        SERIAL PRIMARY KEY,
    bank_id          INTEGER NOT NULL REFERENCES banks(bank_id) ON DELETE CASCADE,
    review_text      TEXT,
    rating           SMALLINT CHECK (rating BETWEEN 1 AND 5),
    review_date      DATE,
    sentiment_label  VARCHAR(20),
    sentiment_score  NUMERIC(6, 4),
    identified_theme VARCHAR(100),
    source           VARCHAR(50) DEFAULT \'Google Play\'
);

-- Indexes for common query patterns
CREATE INDEX IF NOT EXISTS idx_reviews_bank_id  ON reviews(bank_id);
CREATE INDEX IF NOT EXISTS idx_reviews_rating   ON reviews(rating);
CREATE INDEX IF NOT EXISTS idx_reviews_date     ON reviews(review_date);
CREATE INDEX IF NOT EXISTS idx_reviews_sentiment ON reviews(sentiment_label);
'''

os.makedirs('scripts', exist_ok=True)
with open('scripts/schema.sql', 'w') as f:
    f.write(schema_sql)
print("Schema exported to scripts/schema.sql")